In [36]:
%pip install datasets
%pip install biopython

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
import glob
from datasets import load_dataset

# Find all JSONL files in the textbooks/chunk/ directory
jsonl_files = glob.glob("textbooks/chunk/*.jsonl")

# Create data_files dict with all files
data_files = {"train": jsonl_files}

# Load all files into a single dataset
ds = load_dataset("json", data_files=data_files, split="train")

print(f"Total records loaded: {len(ds)}")
print(f"Files loaded: {len(jsonl_files)}")
print(ds[0])


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Total records loaded: 125847
Files loaded: 18
{'id': 'Anatomy_Gray_0', 'title': 'Anatomy_Gray', 'content': 'What is anatomy? Anatomy includes those structures that can be seen grossly (without the aid of magnification) and microscopically (with the aid of magnification). Typically, when used by itself, the term anatomy tends to mean gross or macroscopic anatomy—that is, the study of structures that can be seen without using a microscopic. Microscopic anatomy, also called histology, is the study of cells and tissues using a microscope. Anatomy forms the basis for the practice of medicine. Anatomy leads the physician toward an understanding of a patient’s disease, whether he or she is carrying out a physical examination or using the most advanced imaging techniques. Anatomy is also important for dentists, chiropractors, physical therapists, and all others involved in any aspect of patient treatment that begins with an analysis of clinical signs. The ability to interpret a clinical observ

In [38]:
def parse_title(title: str) -> dict:
    """
    Map an arbitrary textbook chunk title to:
    - textbook_id: normalized textbook identifier, e.g. 'Harrison_Internal_Medicine'
    - specialty: coarse specialty, e.g. 'internal_medicine', 'pharmacology', etc.
    """
    t_low = title.lower()

    textbook_id = None
    specialty = None

    if "anatomy" in t_low:
        textbook_id = "Anatomy_Gray"
        specialty = "anatomy"
    elif "biochemistry" in t_low:
        textbook_id = "Biochemistry_Lippincott"
        specialty = "biochemistry"
    elif "cell" in t_low and "biology" in t_low:
        textbook_id = "Cell_Biology_Alberts"
        specialty = "cell_biology"
    elif "surgery" in t_low or "surgical" in t_low:
        textbook_id = "Surgery_Schwartz"
        specialty = "surgery"
    elif "first aid" in t_low or "step" in t_low:
        textbook_id = "First_Aid_Step1" if "step 1" in t_low else "First_Aid_Step2"
        specialty = "first_aid"
    elif "gynecology" in t_low or "gyneco" in t_low:
        textbook_id = "Gynecology_Novak"
        specialty = "gynecology"
    elif "histology" in t_low:
        textbook_id = "Histology_Ross"
        specialty = "histology"
    elif "immunology" in t_low or "immune" in t_low:
        textbook_id = "Immunology_Janeway"
        specialty = "immunology"
    elif "internal medicine" in t_low or "harriso" in t_low:
        textbook_id = "InternalMed_Harrison"
        specialty = "internal_medicine"
    elif "neurology" in t_low or "neuro" in t_low:
        textbook_id = "Neurology_Adams"
        specialty = "neurology"
    elif "obstetric" in t_low or "obstentric" in t_low:
        textbook_id = "Obstentrics_Williams"
        specialty = "obstetrics"
    elif "pathology" in t_low or "pathoma" in t_low:
        textbook_id = "Pathoma_Husain" if "pathoma" in t_low else "Pathology_Robbins"
        specialty = "pathology"
    elif "pediatric" in t_low or "pediatr" in t_low:
        textbook_id = "Pediatrics_Nelson"
        specialty = "pediatrics"
    elif "pharmacology" in t_low or "pharma" in t_low:
        textbook_id = "Pharmacology_Katzung"
        specialty = "pharmacology"
    elif "physiology" in t_low:
        textbook_id = "Physiology_Levy"
        specialty = "physiology"
    elif "psychiatry" in t_low or "psichiatry" in t_low or "dsm" in t_low:
        textbook_id = "Psichiatry_DSM-5"
        specialty = "psychiatry"

    return {"textbook_id": textbook_id, "specialty": specialty}

In [39]:
def add_metadata(example):
    meta = parse_title(example["title"])
    return {
        "id": example["id"],
        "title": example["title"],
        "text": example["content"],
        **meta,
    }

ds_with_meta = ds.map(add_metadata)


In [40]:
%pip install langchain-community faiss-cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
%pip install langchain-huggingface sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:


from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(
    model_name="pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
)

texts = [r["text"] for r in ds_with_meta]
""" metadatas = [
    {
        "id": r["id"],
        "title": r["title"],
        "textbook_id": r["textbook_id"],
        "specialty": r["specialty"],
    }
    for r in ds_with_meta
] """

# vectorstore = FAISS.from_texts(texts=texts, embedding=embed_model, metadatas=metadatas)
# vectorstore.save_local("faiss_med_textbooks_jsonl")
vectorstore = FAISS.load_local("faiss_med_textbooks_jsonl", embeddings=embed_model, allow_dangerous_deserialization=True)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [44]:
from typing import Optional, List, Dict

def retrieve_textbook_snippets(
    query: str,
    k: int = 5,
    specialty: Optional[str] = None,
    textbook_id: Optional[str] = None,
) -> List[Dict]:
    filters = {}
    if specialty:
        filters["specialty"] = specialty
    if textbook_id:
        filters["textbook_id"] = textbook_id

    docs = vectorstore.similarity_search(
        query,
        k=k,
        filter=filters if filters else None
    )

    return [
        {
            "text": d.page_content,
            "id": d.metadata.get("id"),
            "title": d.metadata.get("title"),
            "textbook_id": d.metadata.get("textbook_id"),
            "specialty": d.metadata.get("specialty"),
        }
        for d in docs
    ]


In [45]:
from typing import Annotated, Optional, List, Dict

def search_textbooks(
    query: Annotated[str, "Clinical questions to search in textbooks"],
    k: Annotated[int, "Number of passages to retrieve"] = 5,
    specialty: Annotated[Optional[str], "Optional specialty filter"] = None,
    textbook_id: Annotated[Optional[str], "Optional textbook identifier"] = None,
) -> List[Dict]:
    """
    Search the JSONL textbook corpus for relevant passages.

    Returns items with:
    - text: snippet content
    - id: original record Id
    - title: original Title string
    - textbook_id: normalized identifier (if detected)
    - specialty: coarse specialty (if detected)
    """
    return retrieve_textbook_snippets(
        query=query,
        k=k,
        specialty=specialty,
        textbook_id=textbook_id,
    )

def search_pharmacology(
    query: Annotated[str, "Question about drugs, doses, interactions, adverse effects"],
    k: int = 5
) -> List[Dict]:
    """Search pharmacology-related textbook content for drug information."""
    return retrieve_textbook_snippets(
        query=query,
        k=k,
        specialty="pharmacology",
    )

def search_radiology(
    query: Annotated[str, "Question about imaging findings or which imaging to order"],
    k: int = 5
) -> List[Dict]:
    """Search imaging-related content in textbooks for radiology guidance."""
    return retrieve_textbook_snippets(
        query=query,
        k=k,
        specialty="radiology",
    )

def search_general_medicine(
    query: Annotated[str, "General medicine question (triage, red flags, basic workup)"],
    k: int = 5
) -> List[Dict]:
    """Search broad, core clinical medicine content in textbooks."""
    return retrieve_textbook_snippets(
        query=query,
        k=k,
        specialty="internal_medicine",  # or use a union of several specialties
    )


In [46]:
from typing import Annotated, Optional, List, Dict
from Bio import Entrez
import time

# Set up Entrez API email (required by NCBI)
Entrez.email = "zsolt.feher@stud.ki.se"  # Replace with your email

def search_pubmed(
    query: Annotated[str, "Medical/clinical search query for PubMed"],
    k: Annotated[int, "Maximum number of articles to retrieve"] = 5,
    min_year: Annotated[Optional[int], "Optional minimum publication year"] = None,
) -> List[Dict]:
    """
    Search PubMed for relevant articles.
    
    Returns items with:
    - pmid: PubMed ID
    - title: Article title
    - authors: List of authors
    - year: Publication year
    - abstract: Article abstract (if available)
    - journal: Journal name
    """
    try:
        # Build search query
        search_term = query
        if min_year:
            search_term += f" AND {min_year}[PDAT] : 3000[PDAT]"
        
        # Search PubMed (use default XML format for Entrez.read() compatibility)
        handle = Entrez.esearch(db="pubmed", term=search_term, retmax=k)
        search_result = Entrez.read(handle)
        handle.close()
        
        # Access ID list from XML response
        id_list = search_result.get("IdList", [])
        
        if not id_list:
            return [{"note": "No articles found for this query"}]
        
        # Fetch article details
        time.sleep(0.5)  # Rate limiting
        handle = Entrez.efetch(db="pubmed", id=",".join(id_list), rettype="xml")
        fetch_result = Entrez.read(handle)
        handle.close()
        
        articles = []
        for article in fetch_result.get("PubmedArticle", []):
            try:
                med_cite = article.get("MedlineCitation", {})
                article_meta = med_cite.get("Article", {})
                
                # Extract authors
                authors = []
                author_list = article_meta.get("AuthorList", [])
                if author_list:
                    for author in author_list:
                        if "LastName" in author:
                            authors.append(author["LastName"])
                
                # Extract PMID
                pmid = med_cite.get("PMID", "N/A")
                if isinstance(pmid, dict):
                    pmid = pmid.get("#text", "N/A")
                
                # Extract year
                pub_date = article_meta.get("Journal", {}).get("JournalIssue", {}).get("PubDate", {})
                year = pub_date.get("Year", "N/A")
                
                # Extract abstract
                abstract_section = article_meta.get("Abstract", {})
                abstract_text = "N/A"
                if isinstance(abstract_section, dict):
                    abstract_list = abstract_section.get("AbstractText", [])
                    if abstract_list:
                        if isinstance(abstract_list, list) and len(abstract_list) > 0:
                            abstract_text = str(abstract_list[0])
                        else:
                            abstract_text = str(abstract_list)
                
                articles.append({
                    "pmid": str(pmid),
                    "title": article_meta.get("ArticleTitle", "N/A"),
                    "authors": authors[:3],  # First 3 authors
                    "year": str(year),
                    "journal": article_meta.get("Journal", {}).get("Title", "N/A"),
                    "abstract": abstract_text,  # Full abstract text
                    "pubmed_url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
                })
            except Exception as article_error:
                print(f"Error parsing article: {article_error}")
                continue
        
        return articles if articles else [{"note": "Could not parse article details"}]
    
    except Exception as e:
        import traceback
        traceback.print_exc()
        return [{"error": f"PubMed search failed: {str(e)}"}]


def search_pubmed_clinical(
    query: Annotated[str, "Clinical question or case to search in PubMed"],
    k: int = 5
) -> List[Dict]:
    """Search PubMed for clinical evidence and case reports."""
    return search_pubmed(query, k=k, min_year=2019)


def search_pubmed_pharmacology(
    query: Annotated[str, "Drug or medication query for PubMed"],
    k: int = 5
) -> List[Dict]:
    """Search PubMed for pharmacology and drug studies."""
    search_term = f"({query}) AND (pharmacology OR drug OR clinical trial OR adverse effect)"
    return search_pubmed(search_term, k=k, min_year=2018)

In [47]:
from typing import Annotated, Optional, List, Dict
from pathlib import Path
import pandas as pd

CASE_ROOT = Path(r"Case_1")

def _safe_read_csv(path: Path, nrows: Optional[int] = None) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, nrows=nrows)
        except Exception as e:
            last_error = e
    raise last_error

def _df_to_text(df: pd.DataFrame, max_rows: int = 20) -> str:
    if df.empty:
        return "CSV is empty."

    clipped = df.head(max_rows).fillna("")
    rows_text = []
    for i, row in clipped.iterrows():
        parts = [f"{col}={row[col]}" for col in clipped.columns]
        rows_text.append(f"Row {i+1}: " + "; ".join(parts))
    return "\n".join(rows_text)

def search_case_csvs(
    user_need: Annotated[str, "What the agent is looking for in the case CSVs"],
    selected_file: Annotated[Optional[str], "Relative or full path of the CSV to open; leave empty to only search"] = None,
    max_candidates: Annotated[int, "Maximum number of candidate CSV files to return"] = 8,
    preview_rows: Annotated[int, "How many rows to preview when converting CSV to text"] = 10,
) -> List[Dict]:
    """
    Search all CSV files under Thesis/Case_1 recursively.
    Mode 1: selected_file is None -> return candidate CSV files for the agent to choose from.
    Mode 2: selected_file is provided -> open that CSV and convert it to text for the conversation.
    """
    if not CASE_ROOT.exists():
        return [{"error": f"Folder not found: {CASE_ROOT.resolve()}"}]

    if selected_file:
        chosen = Path(selected_file)
        

        if not chosen.exists():
            return [{"error": f"Selected file not found: {chosen}"}]

        try:
            df = _safe_read_csv(chosen)
            return [{
                "mode": "opened",
                "file": str(chosen),
                "rows": int(df.shape[0]),
                "columns": list(df.columns),
                "text": _df_to_text(df, max_rows=preview_rows)
            }]
        except Exception as e:
            return [{"error": f"Could not open CSV {chosen}: {str(e)}"}]

    csv_files = list(CASE_ROOT.rglob("*.csv"))
    if not csv_files:
        return [{"error": f"No CSV files found under {CASE_ROOT.resolve()}"}]

    need_tokens = set(user_need.lower().split())
    ranked = []

    for f in csv_files:
        try:
            sample = _safe_read_csv(f, nrows=5)
            cols = [str(c) for c in sample.columns]
            haystack = f"{f.name} {' '.join(cols)}".lower()
            score = sum(1 for t in need_tokens if t in haystack)

            ranked.append({
                "score": score,
                "file": str(f),
                "relative_file": str(f.relative_to(CASE_ROOT)),
                "columns": cols,
                "preview": sample.head(3).fillna("").to_dict(orient="records")
            })
        except Exception as e:
            ranked.append({
                "score": -1,
                "file": str(f),
                "relative_file": str(f.relative_to(CASE_ROOT)),
                "error": str(e)
            })

    ranked.sort(key=lambda x: x["score"], reverse=True)
    return ranked[:max_candidates]

In [48]:
import requests
from typing import Annotated, Dict, List, Any, Optional

RXNAV_BASE = "https://rxnav.nlm.nih.gov/REST"


def _rxnav_get(path: str, **params) -> Dict[str, Any]:
    resp = requests.get(f"{RXNAV_BASE}/{path.lstrip('/')}", params=params, timeout=20)
    resp.raise_for_status()
    return resp.json()


def _flatten_drug_concepts(payload: Dict[str, Any]) -> List[Dict[str, Any]]:
    rows = []
    groups = payload.get("drugGroup", {}).get("conceptGroup", []) or []
    for group in groups:
        fallback_tty = group.get("tty")
        for c in group.get("conceptProperties", []) or []:
            rows.append(
                {
                    "rxcui": c.get("rxcui"),
                    "name": c.get("name"),
                    "synonym": c.get("synonym"),
                    "tty": c.get("tty", fallback_tty),
                    "language": c.get("language"),
                    "suppress": c.get("suppress"),
                    "umlscui": c.get("umlscui"),
                }
            )

    seen = set()
    deduped = []
    for row in rows:
        key = (row.get("rxcui"), row.get("tty"), row.get("name"))
        if key not in seen:
            seen.add(key)
            deduped.append(row)
    return deduped


def _get_related_concepts(rxcui: str, keep_tty: Optional[set] = None) -> List[Dict[str, Any]]:
    payload = _rxnav_get(f"rxcui/{rxcui}/allrelated.json")
    groups = payload.get("allRelatedGroup", {}).get("conceptGroup", []) or []

    rows = []
    for group in groups:
        fallback_tty = group.get("tty")
        for c in group.get("conceptProperties", []) or []:
            row = {
                "rxcui": c.get("rxcui"),
                "name": c.get("name"),
                "tty": c.get("tty", fallback_tty),
                "synonym": c.get("synonym"),
            }
            if keep_tty is None or row["tty"] in keep_tty:
                rows.append(row)

    seen = set()
    deduped = []
    for row in rows:
        key = (row.get("rxcui"), row.get("tty"), row.get("name"))
        if key not in seen:
            seen.add(key)
            deduped.append(row)
    return deduped


def _detect_mode(query: str) -> str:
    q = query.strip().replace("-", "")
    if q.isdigit() and len(q) in {10, 11}:
        return "ndc"
    if q.isdigit():
        return "rxcui"
    return "name"


def search_rxnorm(
    query: Annotated[str, "Drug name, NDC, or RxCUI to look up in RxNorm."],
    search_by: Annotated[str, "auto, name, ndc, or rxcui."] = "auto",
    max_results: Annotated[int, "Maximum number of primary results to return."] = 5,
    include_related: Annotated[bool, "Include related brand/generic/dose-form concepts when possible."] = True,
) -> Dict[str, Any]:
    """
    Query RxNorm / RxNav for medication normalization.
    Best for pharmacy review, medication reconciliation, and NDC-to-RxCUI verification.
    """
    mode = _detect_mode(query) if search_by == "auto" else search_by.lower()

    if mode == "name":
        payload = _rxnav_get("drugs.json", name=query)
        candidates = _flatten_drug_concepts(payload)[:max_results]

        result = {
            "query": query,
            "search_by": "name",
            "candidate_count": len(candidates),
            "candidates": candidates,
        }

        if include_related and candidates:
            top_rxcui = candidates[0].get("rxcui")
            if top_rxcui:
                result["related_concepts"] = _get_related_concepts(
                    top_rxcui,
                    keep_tty={"IN", "PIN", "MIN", "BN", "SCD", "SBD", "GPCK", "BPCK"},
                )[:20]

        return result

    if mode == "ndc":
        payload = _rxnav_get("ndcstatus.json", ndc=query.replace("-", ""))
        ndc_status = payload.get("ndcStatus", {}) or {}
        rxcui = ndc_status.get("rxcui")

        result = {
            "query": query,
            "search_by": "ndc",
            "ndc_status": ndc_status,
        }

        if include_related and rxcui:
            result["related_concepts"] = _get_related_concepts(
                rxcui,
                keep_tty={"IN", "PIN", "MIN", "BN", "SCD", "SBD", "GPCK", "BPCK"},
            )[:20]

        return result

    if mode == "rxcui":
        related = _get_related_concepts(
            query,
            keep_tty={"IN", "PIN", "MIN", "BN", "SCD", "SBD", "GPCK", "BPCK"},
        )
        return {
            "query": query,
            "search_by": "rxcui",
            "candidate_count": len(related),
            "related_concepts": related[: max_results if max_results > 0 else 20],
        }

    return {
        "query": query,
        "search_by": mode,
        "error": "search_by must be one of: auto, name, ndc, rxcui",
    }

In [ ]:
from autogen import ConversableAgent, LLMConfig


llm = {
    "config_list": [
        {
            "api_type": "openai",
            "model": "openai/gpt-oss-20b",
            "base_url": "http://localhost:1234/v1",
            "api_key": "lm-studio",
        },
    ],
    "cache_seed": None,  # Disable caching.
}

lvm = {
    "config_list": [
        {
            "api_type": "openai",
            "model": "hulu-med-flash-preview-27b",
            "base_url": "http://localhost:1234/v1",
            "api_key": "lm-studio",
            "timeout": 2400,
        },
    ],
    "cache_seed": None,  # Disable caching.
}

manager_llm = {
    "config_list": [
        {
            "api_type": "openai",
            "model": "medgemma-4b-it",
            "base_url": "http://localhost:1234/v1",
            "api_key": "lm-studio",
        },
    ],
    "cache_seed": None,  # Disable caching.
}


senior_doctor = ConversableAgent(
    name="senior_doctor",
    llm_config=manager_llm,
    max_consecutive_auto_reply=7,
    system_message=(
            "You are a senior senior physician, and your are responsible to oversee the treatment planning. You don't act outside of your domain nor doing any action, only plan a treamtent course and use the registered tools. First use search_case_csvs to find and open relevant case data about the current patient, so you can cunstuct the context. If you want to open it, then insert the path of the file to the function's 'selected_file' argument and specify what you need on user_need."
            "The files that name starts with 'd_' are diagnosis code dictionaries, so they are only for reference and are not the patient's records."
            " Pay attention to the dates in these records to understand the timeline of the case. "
            "Before proposing diagnoses or treatments, call a PubMed search tool or textbook search tool with relevant clinical questions to gather evidence. "
            "to retrieve supporting evidence, then reason based on that evidence." \
            "You are working with a team of specialists (pharma_specialist, radiologist, general_physician) who can provide input on specific aspects like drug information, imaging interpretation, and primary care management so collaborate with them by asking them to search for relevant information and incorporating their findings into your reasoning process. " \
            "When you have a final recommendation or conclusion, include the word 'DONE!' in your message to indicate that you have completed your reasoning process and are ready for termination."
        ),
    is_termination_msg=lambda x: "DONE!" in (x.get("content", "") or "").upper(),
)

senior_doctor.register_for_llm(description="Search textbooks for clinical information")(search_textbooks)
senior_doctor.register_for_llm(description="Search pharmacology resources")(search_pharmacology)
senior_doctor.register_for_llm(description="Search radiology resources")(search_radiology)
senior_doctor.register_for_llm(description="Search general medicine resources")(search_general_medicine)
senior_doctor.register_for_llm(description="Search PubMed for clinical evidence")(search_pubmed_clinical)
senior_doctor.register_for_llm(description="Search all CSV files under /Case_1 recursively, choose the most relevant one, and open it as text.")(search_case_csvs)


senior_doctor.register_for_execution()(search_textbooks)
senior_doctor.register_for_execution()(search_pharmacology)
senior_doctor.register_for_execution()(search_radiology)
senior_doctor.register_for_execution()(search_general_medicine)
senior_doctor.register_for_execution()(search_pubmed_clinical)
senior_doctor.register_for_execution()(search_case_csvs)
senior_doctor.description = "Senior doctor with access to textbook and PubMed search tools for evidence-based reasoning"


pharma_specialist = ConversableAgent(
        name="pharma_specialist",
        llm_config=llm,
        max_consecutive_auto_reply=10,
        is_termination_msg=lambda x: "DONE!" in (x.get("content", "") or "").upper(),
        system_message=(
            """You are a clinical pharmacology specialist, and you are responsible of the medicine and drug planning.  You don't act outside of your domain nor doing any action, only plan a treamtent course and use the registered tools. First use search_case_csvs to find and open relevant case data about the current patient, so you can cunstuct the context.  If you want to open it, then insert the path of the file to the function's 'selected_file' argument. Pay attention to the dates in these records to understand the timeline of the case.  
            The files that name starts with 'd_' are diagnosis code dictionaries, so they are only for reference and are not the patient's records.
            Also use search_pubmed_pharmacology, search_pharmacology or search_textbooks to retrieve drug information from textbooks before recommending or critiquing any medication.
            Description: Conduct a comprehensive review of the prescribed medications for the patient with the symptoms gathered from the case CSVs.:

Consider their medical history, current condition, and the emergency physician’s treatment plan.
Your review should include:
1. A thorough analysis of each prescribed medication using the RxNorm tool
2. Potential drug interactions, including severity and clinical significance
3. Dose appropriateness considering the patient’s condition, age, weight, and renal/hepatic function
4. Any contraindications based on the patient’s medical history or current condition
5. Identification of any high-alert medications that require special handling or monitoring
6. Recommendations for medication adjustments, alternatives, or additional monitoring if necessary
7. Important side effects or adverse reactions to watch for in the emergency setting
8. Specific administration instructions or precautions for the nursing staff
9. Any drug-disease interactions that could impact the patient’s current condition
Use the RxNorm tool extensively to gather detailed information about each medication.
Use the search tool to find the latest pharmacological guidelines or information on rare drug effects if necessary.
Expected Output: Provide a detailed medication safety report including:
1. List of prescribed medications with a comprehensive analysis of each
2. Identified drug interactions, contraindications, or concerns
3. Dose appropriateness evaluations and any recommended adjustments
4. Specific administration instructions and precautions
5. Potential adverse effects to monitor in the emergency setting
6. Recommendations for medication therapy optimization
7. Any additional pharmacological considerations relevant to the patient’s care
Format your response as follows:
MEDICATION SAFETY REPORT
Patient Condition Summary: [Brief overview of relevant clinical information]
Prescribed Medications Analysis:
1. [Drug Name] (RxNorm verified):
- Dose/Route/Frequency: [As prescribed]
- Indication: [For what condition]
- Appropriateness: [Comment on dose, considering patient factors]
- Interactions: [List any significant interactions]
- Contraindications: [If any, based on patient’s condition]
- Administration Instructions: [Specific guidance for nursing]
- Monitoring: [What to watch for, including adverse effects]
2. [Repeat for each medication]
Overall Medication Therapy Assessment:
- Drug-Disease Interactions: [Any concerns with patient’s conditions]
- High-Alert Medications: [Identify any that require special precautions]
- Pharmacokinetic Considerations: [Any adjustments needed for renal/hepatic function]
Recommendations:
1. [Specific recommendation for medication adjustment, monitoring, or alternative]
2. [Additional recommendations as needed]
Emergency Pharmacology Considerations:
- [Any special considerations for medication use in the ER setting]
References:
- [List any guidelines or resources used from the search tool, if applicable]
            """
        ),
)

pharma_specialist.register_for_llm(description="Search pharmacology resources")(search_pharmacology)
pharma_specialist.register_for_llm(description="Search textbooks for clinical information")(search_textbooks)
pharma_specialist.register_for_llm(description="Search PubMed for drug research")(search_pubmed_pharmacology)
pharma_specialist.register_for_llm(description="Search all CSV files under /Case_1 recursively, choose the most relevant one, and open it as text.")(search_case_csvs)
pharma_specialist.register_for_llm(description="Lookup medications in RxNorm/RxNav by drug name, NDC, or RxCUI. Returns normalized concepts and related brand/generic forms.")(search_rxnorm)

pharma_specialist.register_for_execution()(search_pubmed_pharmacology)
pharma_specialist.register_for_execution()(search_pharmacology)
pharma_specialist.register_for_execution()(search_textbooks)
pharma_specialist.description = "Pharmacology specialist agent with access to textbook and PubMed search tools for drug information retrieval"
pharma_specialist.register_for_execution()(search_case_csvs)
pharma_specialist.register_for_execution()(search_rxnorm)


general_physician = ConversableAgent(
        name="general_physician",
        max_consecutive_auto_reply=10,
        is_termination_msg=lambda x: "DONE!" in (x.get("content", "") or "").upper(),
        llm_config=llm,
        system_message=(
            """You are a general physician in primary care. You are responsible of assessing the physology of the patient and plan accordingly.  You don't act outside of your domain nor doing any action, only plan a treamtent course and use the registered tools First use search_case_csvs to find and open relevant case data about the current patient, so you can cunstuct the context.  If you want to open it, then insert the path of the file to the function's 'selected_file' argument. Pay attention to the dates in these records to understand the timeline of the case. 
            The files that name starts with 'd_' are diagnosis code dictionaries, so they are only for reference and are not the patient's records.
            Also use search_pubmed_clinical to support triage, workup, and follow-up decisions. Provide a comprehensive diagnosis and treatment plan for the patient with the symptoms you find in the case CSVs.:
            
            Your assessment should include:
            1. A thorough analysis of the patient’s symptoms, vital signs, and medical history
            2. Differential diagnoses, listing the most likely conditions in order of probability
            3. Recommended diagnostic tests or imaging studies, with justification for each
            4. A detailed initial treatment plan, including medications, procedures, and interventions
            5. Consideration of potential complications and how to mitigate them
            6. Any necessary consultations with specialists, with clear reasons for each referral
            7. A plan for ongoing monitoring and reassessment of the patient’s condition
            Use the search tool to find the latest evidence-based guidelines or unusual clinical presentations if necessary.
            Expected Output: Provide a comprehensive medical report including:
            1. Primary working diagnosis with supporting evidence
            2. Differential diagnoses in order of likelihood, with brief explanations
            3. Detailed treatment plan, including:
            a. Medications (dosage, route, frequency) verified with RxNorm
            b. Immediate interventions or procedures
            c. Fluid management if applicable
            d. Pain management strategy
            4. Diagnostic tests ordered, with justification for each
            5. Potential complications to monitor for, with specific warning signs
            6. Consultations requested, if any, with clear rationale
            7. Plan for ongoing monitoring and criteria for reassessment
            8. Relevant evidence-based guidelines or literature referenced (if search tool was used)
            Format your response as follows:
            PRIMARY DIAGNOSIS: [State the most likely diagnosis]
            Supporting Evidence: [List key findings that support this diagnosis]
            Differential Diagnoses:
            1. [Diagnosis 1]: [Brief explanation]
            2. [Diagnosis 2]: [Brief explanation]
            3. [Diagnosis 3]: [Brief explanation]
            Treatment Plan:
            1. Medications:
            a. [Drug name, dosage, route, frequency] - [Purpose] (RxNorm verified)
            b. [Drug name, dosage, route, frequency] - [Purpose] (RxNorm verified)
            2. Interventions/Procedures: [List and describe]
            3. Fluid Management: [If applicable]
            4. Pain Management: [Strategy]
            Diagnostic Tests:
            1. [Test name]: [Justification]
            2. [Test name]: [Justification]
            Potential Complications:
            1. [Complication]: [Warning signs and management plan]
            2. [Complication]: [Warning signs and management plan]
            Consultations:
            1. [Specialist]: [Rationale for consultation]
            Monitoring Plan: [Describe ongoing monitoring and reassessment criteria]
            Evidence-Based Guidelines: [Reference any guidelines or literature used, if applicable]"""
                    ),
            )

general_physician.register_for_llm(description="Search general medicine resources")(search_general_medicine)
general_physician.register_for_llm(description="Search textbooks for clinical information")(search_textbooks)
general_physician.register_for_llm(description="Search PubMed for clinical evidence")(search_pubmed_clinical)
general_physician.register_for_llm(description="Search all CSV files under /Case_1 recursively, choose the most relevant one, and open it as text.")(search_case_csvs)
general_physician.register_for_llm(description="Lookup medications in RxNorm/RxNav by drug name, NDC, or RxCUI.")(search_rxnorm)

general_physician.register_for_execution()(search_general_medicine)
general_physician.register_for_execution()(search_textbooks)
general_physician.register_for_execution()(search_pubmed_clinical)
general_physician.register_for_execution()(search_rxnorm)
general_physician.description = "General physician agent with access to textbook and PubMed search tools for primary care decision support"


radiologist = ConversableAgent(
        name="radiologist",
        llm_config=manager_llm,
        max_consecutive_auto_reply=3,
        
        system_message=(
            "You are a radiologist, and your are  responsible of interpeting the imaging results. You don't act outside of your domain nor doing any action, only plan a treamtent course and use the registered tools Use search_radiology to justify imaging "
            "interpretations and recommendations from textbooks. "
        ),
)

radiologist.register_for_llm(description="Search radiology resources")(search_radiology)
radiologist.register_for_llm(description="Search textbooks for clinical information")(search_textbooks)

radiologist.register_for_execution()(search_radiology)
radiologist.register_for_execution()(search_textbooks)
general_physician.description = "Radiologist agent with access to textbook and radiology-specific search tools for imaging interpretation support"


In [50]:
from autogen import GroupChat, GroupChatManager
# All medical agents participate in the group chat
agents = [senior_doctor, pharma_specialist, general_physician, radiologist]

groupchat = GroupChat(
    max_round=20,
    agents=agents,
    messages=[],
    speaker_selection_method="auto",  # manager uses an LLM to pick who speaks next
    send_introductions=True
)

manager = GroupChatManager(
    name="treatment_planning_manager",
    groupchat=groupchat,
    human_input_mode="NEVER",
    llm_config=manager_llm,
    system_message=(
        "You are a manager agent overseeing a group chat of medical specialists collaborating on a complex patient case. "
        "Your role is to facilitate effective communication, ensure that the agents gather necessary information from their tools, and guide the conversation towards a comprehensive treatment plan. "
        "Use your understanding of the case and the agents' expertise to select who should speak next and when to end the discussion with a final recommendation. Make sure that every agent contributes their insights and that the senior doctor leads the reasoning process based on the evidence gathered by all specialists. However, give space for the senior doctor to take the lead in synthesizing the information and proposing the final plan and terminating the conversation by saying 'DONE!'."
    )
)

In [52]:
# New agent to analyze the image and start groupchat
from autogen import ConversableAgent
import base64
from pathlib import Path

image_analyzer = ConversableAgent(
    name="image_analyzer",
    llm_config=manager_llm,  # Assuming lvm supports vision
    system_message="You are an expert radiologist. Analyze the provided chest X-ray image and provide a detailed description of the findings, including any abnormalities, as you would in a radiology report.",
    max_consecutive_auto_reply=1,
)

# Path to the image
image_path = r"C:\Users\zsolf\Desktop\Thesis\Case_1\CXR-JPG\s52435728\9b00bfd9-c8d1871a-74623a5d-73f2f0b1-f13aa0ad.jpg"


b64 = base64.b64encode(Path(image_path).read_bytes()).decode("utf-8")
LOCAL_IMAGE_DATA_URI = f"data:image/png;base64,{b64}"

# Create message with image
message_with_image = [
    {"type": "text", "text": """Interpret the given chest X-ray image or description to identify and explain any abnormalities, patterns, or notable findings relevant to medical diagnosis.

Provide a detailed analysis that includes:
- Identification of anatomical structures visible in the X-ray.
- Detection of any abnormalities.
- Possible clinical implications or differential diagnoses based on the findings.
- If image data is not provided, interpret the described findings accurately.

# Steps
1. Review the chest X-ray image or provided descriptive data carefully.
2. Systematically examine anatomical landmarks: lungs, heart silhouette, diaphragm, bony thorax.
3. Identify any abnormal radiographic signs or patterns.
4. Correlate findings with common chest pathologies.
5. Offer a concise, clear diagnostic impression.

# Output Format
Provide the interpretation in a medically professional, clear, and structured format, such as:

**Chest X-Ray Interpretation:**
- **Anatomical Structures:** [description]
- **Findings:** [detailed findings]
- **Impression:** [diagnostic summary and possible differential diagnoses]

# Notes
- Avoid overdiagnosis without clear evidence.
- Base interpretation only on provided data.
- Highlight any need for further imaging or clinical correlation if necessary. """},
    {"type": "image_url", "image_url": {"url": LOCAL_IMAGE_DATA_URI}}
]

# Generate the description
response = image_analyzer.generate_oai_reply(
    messages=[{"role": "user", "content": message_with_image}],
    sender=None
)

radiologist_description = response[1]["content"] if response[0] else "Unable to analyze image."

print("Radiologist Description:")
print(radiologist_description)

# Use the description as patient complaint
patient_complaint = f"Patient presents with the following chest X-ray findings from the Radiologist agent, please review the patient files and discuss the treatment plan : {radiologist_description}"

# Start the groupchat
general_physician.initiate_chat(
    recipient=manager,
    message=patient_complaint,
    max_round=30,
    clear_history=True,
)

[autogen.oai.client: 04-30 16:21:30] {744} WARNING - Model medgemma-4b-it is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
Radiologist Description:
**Chest X-Ray Interpretation:**

*   **Anatomical Structures:** The image shows the heart silhouette, mediastinum, lungs (including visible portions of both upper lobes and lower lobes), and diaphragm. The ribs are also discernible.

*   **Findings:** There is increased opacity in the left lung field, particularly in the lower lobe. This could represent consolidation due to pneumonia or other infiltrative processes. The right lung appears relatively clear. The heart size is mildly enlarged (cardiomegaly). No obvious pleural effusion is seen on this single view.

*   **Impression:** Left lower lobe opacity concerning for possible pneumonia or another process causing consolidation. Mild cardiomegaly. Further evaluation with lateral decubit

ChatResult(chat_id=276899296951375254864455689334143768496, chat_history=[{'content': 'Patient presents with the following chest X-ray findings from the Radiologist agent, please review the patient files and discuss the treatment plan : **Chest X-Ray Interpretation:**\n\n*   **Anatomical Structures:** The image shows the heart silhouette, mediastinum, lungs (including visible portions of both upper lobes and lower lobes), and diaphragm. The ribs are also discernible.\n\n*   **Findings:** There is increased opacity in the left lung field, particularly in the lower lobe. This could represent consolidation due to pneumonia or other infiltrative processes. The right lung appears relatively clear. The heart size is mildly enlarged (cardiomegaly). No obvious pleural effusion is seen on this single view.\n\n*   **Impression:** Left lower lobe opacity concerning for possible pneumonia or another process causing consolidation. Mild cardiomegaly. Further evaluation with lateral decubitus films a